In [1]:
import os
import gc
import random
import warnings
from tqdm import tqdm
from PIL import Image

import numpy as np 
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import torch
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.models import resnet50, ResNet50_Weights

In [2]:
def set_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    
    warnings.filterwarnings("ignore")
    random.seed(seed)
    np.random.seed(seed)
    
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) 
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(False)
    
    print(f"Random seed set to {seed}")

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except:
        pass

In [3]:
RANDOM_SEED = 42
set_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Random seed set to 42
Device: cuda


In [4]:
CRC100K_PATH = "/kaggle/input/datasets/cryandrrich/nckh2026/NCT-CRC-HE-100K/NCT-CRC-HE-100K"
PATHMNIST_PATH = "/kaggle/input/datasets/cryandrrich/nckh2026/pathmnist_224.npz"
HISTOSET_PATH = "/kaggle/input/datasets/cryandrrich/nckh2026/HistoSet-5x14/HistoSet-5x14"
SKINTISSUE_PATH = "/kaggle/input/datasets/cryandrrich/nckh2026/SkinTissue/SkinTissue/tiles"

BATCH_SIZE = 256
IMAGE_SIZE = (224, 224)

BUDGET = 5000
MAX_ITERATIONS = 10

In [5]:
class NPZDataset(Dataset):
    def __init__(self, npz_path, split="train", transform=None):
        data = np.load(npz_path)
        
        if split == "train":
            self.img = np.concatenate((data["train_images"], data["val_images"]), axis=0)
            self.lbl = np.concatenate((data["train_labels"], data["val_labels"]), axis=0).squeeze()
        elif split == "test":
            self.img = data["test_images"]
            self.lbl = data["test_labels"].squeeze()
        else:
            raise ValueError("Split must be 'train' or 'test'")
            
        self.transform = transform
        
        self.classes = [
            "adipose", "background", "debris", "lymphocytes", "mucus", 
            "smooth_muscle", "normal_colon_mucosa", "cancer_associated_stroma", 
            "colorectal_adenocarcinoma"
        ]

    def __len__(self):
        return len(self.img)

    def __getitem__(self, idx):
        img = self.img[idx]
        label = self.lbl[idx]
            
        img = Image.fromarray(img)
            
        if self.transform:
            img = self.transform(img)
            
        return img, label

In [6]:
def get_data_loaders(data_path, 
                     batch_size=BATCH_SIZE, 
                     image_size=IMAGE_SIZE,
                     seed=RANDOM_SEED):
    transform = transforms.Compose([
        transforms.Resize(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    num_cores = min(4, os.cpu_count() or 4) 
    
    if data_path.endswith(".npz"):
        train_dataset = NPZDataset(npz_path=data_path, split="train", transform=transform)
        test_dataset = NPZDataset(npz_path=data_path, split="test", transform=transform)
        class_names = train_dataset.classes
    else:
        full_dataset = ImageFolder(root=data_path, transform=transform)
        class_names = full_dataset.classes
        
        total_size = len(full_dataset)
        train_size = int(0.8 * total_size)
        test_size = total_size - train_size
        
        generator = torch.Generator().manual_seed(seed)
        train_dataset, test_dataset = random_split(
            full_dataset, 
            [train_size, test_size], 
            generator=generator
        )

    print(f"Train size: {len(train_dataset)} | Test size: {len(test_dataset)}")

    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=False,
        num_workers=num_cores,     
        pin_memory=True,        
        prefetch_factor=2,         
        persistent_workers=False
    )
    
    test_loader = DataLoader(
        test_dataset, 
        batch_size=batch_size, 
        shuffle=False,
        num_workers=num_cores,     
        pin_memory=True,        
        prefetch_factor=2,         
        persistent_workers=False
    )
    
    return train_loader, test_loader, class_names

In [7]:
def extract_embeddings(dataloader, model, device=DEVICE):
    model = model.to(device)
    model.eval()

    all_embeddings = []
    true_labels = []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Extracting Embeddings"):
            images = images.to(device, non_blocking=True) 
            
            features = model(images)
            
            all_embeddings.append(features.cpu().numpy())
            true_labels.append(labels.numpy())

    embeddings = np.vstack(all_embeddings)
    true_labels = np.concatenate(true_labels)
    
    del all_embeddings
    clear_memory()
    
    return embeddings, true_labels

In [8]:
resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
resnet.fc = torch.nn.Identity()

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 168MB/s] 


In [9]:
train_mnist_loader, test_mnist_loader, class_mnist_names = get_data_loaders(PATHMNIST_PATH)

train_mnist_embeddings, train_mnist_labels = extract_embeddings(train_mnist_loader, resnet)
test_mnist_embeddings, test_mnist_labels = extract_embeddings(test_mnist_loader, resnet)

del train_mnist_loader, test_mnist_loader, class_mnist_names
del train_mnist_embeddings, train_mnist_labels
del test_mnist_embeddings, test_mnist_labels
clear_memory()

Train size: 100000 | Test size: 7180


Extracting Embeddings: 100%|██████████| 29/29 [00:25<00:00,  1.13it/s]


In [10]:
train_histo_loader, test_histo_loader, class_histo_names = get_data_loaders(HISTOSET_PATH)

train_histo_embeddings, train_histo_labels = extract_embeddings(train_histo_loader, resnet)
test_histo_embeddings, test_histo_labels = extract_embeddings(test_histo_loader, resnet)

del train_histo_loader, test_histo_loader, class_histo_names
del train_histo_embeddings, train_histo_labels
del test_histo_embeddings, test_histo_labels
clear_memory()

Train size: 22400 | Test size: 5600


Extracting Embeddings: 100%|██████████| 22/22 [00:28<00:00,  1.28s/it]
